# 04a · Spark MLlib model: logistic regression

**Task.** Binary classification: predict whether a driver files a car insurance claim (`OUTCOME = 1`) from demographic, vehicle and driving-history features.

**Why logistic regression.** It is one of the two models from Phase 2, so there is a scikit-learn result to compare against. It outputs a claim probability, which is what an insurer needs for pricing, and its coefficients convert to odds ratios that can be explained feature by feature. Phase 2 also found it performed as well as XGBoost on this dataset.

**Inputs**

- `workspace.car_insurance.claims_gold_features` (from `03_gold_layer`)
- `workspace.car_insurance.claims_model_split` (from `03b_delta_tables`)

**Outputs**

- Fitted pipeline saved to `/Volumes/workspace/car_insurance/models/logistic_regression`
- Test-set metrics and the comparison with Phase 2, shown below

**Assumptions**

- Notebooks 01 to 03b have been run.
- Serverless compute with **environment version 4 or later** (Environment panel on the right of the notebook), which is what enables Spark ML on serverless.
- Every column of the Gold feature table other than `ID` and `OUTCOME` is a model input. Text columns are treated as categories and numeric columns as numbers.

In [0]:
%run ./00_setup

# 00 · Shared setup: storage, table names and Delta helpers

**Purpose.** One place for everything the other notebooks share: where the data lives, what each Delta table is called, how tables are written and verified, and how models are split and scored. Keeping it here means all team members write to the same tables in the same way.

**How it is used.** This notebook is not run on its own. Every other notebook starts with a cell containing only `%run ./00_setup`, which runs these cells and makes the names and functions below available.

**Inputs.** None.

**Outputs.** No tables. It creates the schema `workspace.car_insurance` and the volumes `raw` (source files) and `models` (saved Spark ML models) if they do not exist yet.

**Assumptions**

- The catalog is named `workspace` (the Databricks Free Edition default). If yours differs, change `CATALOG` below; nothing else needs editing.
- Serverless compute.

**What it provides**

| Name | Use |
|---|---|
| `BRONZE_TABLE`, `SILVER_TABLE`, `GOLD_FEATURES_TABLE`, `GOLD_SUMMARY_TABLE`, `SPLIT_TABLE` | Full table names. Always use these instead of typing a table name |
| `CLAIMS_CSV`, `SPLIT_CSV`, `MODEL_DIR` | Paths inside the volumes |
| `require_source_files(...)` | Stops with a clear message if an uploaded file is missing |
| `write_delta(df, table_name)` | Saves a DataFrame as a Delta table |
| `validate_tables({...})` | Reads tables back and checks format, rows, columns and key |
| `load_model_splits()` | Train, validation and test DataFrames with the Phase 2 split |
| `evaluate_classifier(predictions)` | Accuracy, precision, recall, F1, ROC-AUC and confusion counts |

Setup loaded: tables in workspace.car_insurance, files in /Volumes/workspace/car_insurance


## 1. Data and split

The split is the Phase 2 split, read from the reference table: models are fitted on **train**, the regularization strength is chosen on **validation**, and **test** is used once at the end.

In [0]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import Imputer, MinMaxScaler, OneHotEncoder, StringIndexer, VectorAssembler

NUMERIC_TYPES = ("tinyint", "smallint", "int", "bigint", "float", "double", "decimal", "boolean")
ALWAYS_CATEGORICAL = {"POSTAL_CODE"}   # an area label, not a quantity

gold_types = dict(spark.table(GOLD_FEATURES_TABLE).drop("split").dtypes)
candidates = [c for c in gold_types if c not in (ID_COLUMN, LABEL_COLUMN) and not c.startswith("_")]

CATEGORICAL_FEATURES = [c for c in candidates
                        if gold_types[c] == "string" or c in ALWAYS_CATEGORICAL]
NUMERIC_FEATURES = [c for c in candidates
                    if c not in CATEGORICAL_FEATURES and gold_types[c].startswith(NUMERIC_TYPES)]
ignored = [c for c in candidates if c not in CATEGORICAL_FEATURES + NUMERIC_FEATURES]

print(f"Categorical features ({len(CATEGORICAL_FEATURES)}):", CATEGORICAL_FEATURES)
print(f"Numeric features ({len(NUMERIC_FEATURES)}):", NUMERIC_FEATURES)
if ignored:
    print("Ignored (unsupported type):", ignored)


def prepare(df):
    """Type alignment only: label and numbers as double, categories as text. Nothing is fitted."""
    return df.select(
        ID_COLUMN,
        F.col(LABEL_COLUMN).cast("double").alias("label"),
        *[F.col(c).cast("string").alias(c) for c in CATEGORICAL_FEATURES],
        *[F.col(c).cast("double").alias(c) for c in NUMERIC_FEATURES],
    )


train_df, validation_df, test_df = (prepare(split) for split in load_model_splits())

Categorical features (10): ['AGE', 'GENDER', 'RACE', 'DRIVING_EXPERIENCE', 'EDUCATION', 'INCOME', 'VEHICLE_YEAR', 'POSTAL_CODE', 'VEHICLE_TYPE', 'CAR_AGE']
Numeric features (10): ['CREDIT_SCORE', 'VEHICLE_OWNERSHIP', 'MARRIED', 'CHILDREN', 'ANNUAL_MILEAGE', 'SPEEDING_VIOLATIONS', 'DUIS', 'PAST_ACCIDENTS', 'INSURANCE_RISK', 'FAMILY']


In [0]:
split_stats = [(name, split.agg(F.count("*").alias("drivers"), F.sum("label").alias("claims")).first())
               for name, split in zip(SPLIT_NAMES, (train_df, validation_df, test_df))]

display(spark.createDataFrame(
    [(name, int(row["drivers"]), int(row["claims"]), round(row["claims"] / row["drivers"], 4))
     for name, row in split_stats],
    "split string, drivers long, claims long, claim_rate double"))

N_TRAIN = int(split_stats[0][1]["drivers"])

split,drivers,claims,claim_rate
train,6400,1982,0.3097
validation,1600,495,0.3094
test,2000,656,0.328


In [0]:
# Missing values in the training rows decide which preprocessing is needed.
train_missing = train_df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in CATEGORICAL_FEATURES + NUMERIC_FEATURES]
).first().asDict()

display(spark.createDataFrame(
    [(c, "categorical" if c in CATEGORICAL_FEATURES else "numeric", int(train_missing[c]))
     for c in CATEGORICAL_FEATURES + NUMERIC_FEATURES],
    "feature string, kind string, missing_in_train long"))

missing_categories = [c for c in CATEGORICAL_FEATURES if train_missing[c]]
assert not missing_categories, f"Categorical features with missing values need a Silver rule: {missing_categories}"

feature,kind,missing_in_train
AGE,categorical,0
GENDER,categorical,0
RACE,categorical,0
DRIVING_EXPERIENCE,categorical,0
EDUCATION,categorical,0
INCOME,categorical,0
VEHICLE_YEAR,categorical,0
POSTAL_CODE,categorical,0
VEHICLE_TYPE,categorical,0
CAR_AGE,categorical,0


## 2. Pipeline

All preprocessing is part of one Spark ML `Pipeline`, so calling `fit` on the training rows fits every step on training data only, and the saved model carries its own preprocessing.

| Stage | What it does | Fitted on train | Phase 2 equivalent |
|---|---|---|---|
| `StringIndexer` | Maps each category label to an index, in reverse alphabetical order | Label list | `pd.get_dummies` |
| `OneHotEncoder` (`dropLast`) | One 0/1 column per category, dropping the last index | Category count | `drop_first=True` |
| `Imputer` (median) | Fills missing numeric values | Medians | Linear-regression imputation |
| `VectorAssembler` | Combines all inputs into one feature vector | Nothing | Column selection |
| `MinMaxScaler` | Rescales every feature to 0–1 | Min and max | `MinMaxScaler` |
| `LogisticRegression` | L2-penalized logistic regression | Coefficients | `LogisticRegression` (saga, L2) |

**Reference categories.** Indexing in reverse alphabetical order and dropping the last index removes the alphabetically first label of each feature. That is the same category `drop_first=True` removed in Phase 2, so coefficients and odds ratios are measured against the same reference groups.

**Regularization.** scikit-learn's `C` and Spark's `regParam` are the same penalty written two ways: `regParam = 1 / (C × n_train)`. Spark's internal standardization is switched off because the features are already on a 0–1 scale, as in Phase 2; leaving it on would change which coefficients the penalty shrinks.

In [0]:
def build_pipeline(c_value):
    """Preprocessing + logistic regression, with the penalty given as scikit-learn's C."""
    indexed = [f"{c}_index" for c in CATEGORICAL_FEATURES]
    encoded = [f"{c}_onehot" for c in CATEGORICAL_FEATURES]
    imputed = [f"{c}_imputed" for c in NUMERIC_FEATURES]

    return Pipeline(stages=[
        # handleInvalid="error": a category never seen in training should stop the run, not be guessed.
        StringIndexer(inputCols=CATEGORICAL_FEATURES, outputCols=indexed,
                      stringOrderType="alphabetDesc", handleInvalid="error"),
        OneHotEncoder(inputCols=indexed, outputCols=encoded, dropLast=True),
        Imputer(inputCols=NUMERIC_FEATURES, outputCols=imputed, strategy="median"),
        VectorAssembler(inputCols=encoded + imputed, outputCol="features_unscaled"),
        MinMaxScaler(inputCol="features_unscaled", outputCol="features"),
        LogisticRegression(featuresCol="features", labelCol="label",
                           regParam=1.0 / (c_value * N_TRAIN), elasticNetParam=0.0,
                           standardization=False, maxIter=200),
    ])

## 3. Choosing the regularization strength

Phase 2 searched 13 values of `C` from 0.001 to 1000 with 5-fold cross-validation and picked the one with the best accuracy. Here every other value of that grid (7 values) is fitted on the training split and scored on the validation split, again by accuracy. A single validation split is used instead of cross-validation to keep the number of Spark fits small on shared serverless compute. When two values tie, the smaller `C` (the simpler model) is kept.

In [0]:
C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

tuning_rows, best = [], None
for c_value in C_GRID:
    candidate = build_pipeline(c_value).fit(train_df)
    train_metrics = evaluate_classifier(candidate.transform(train_df), "label")
    validation_metrics = evaluate_classifier(candidate.transform(validation_df), "label")
    tuning_rows.append((c_value, 1.0 / (c_value * N_TRAIN),
                        round(train_metrics["accuracy"], 4),
                        round(validation_metrics["accuracy"], 4),
                        round(validation_metrics["roc_auc"], 4)))
    if best is None or validation_metrics["accuracy"] > best["validation_accuracy"]:
        best = {"C": c_value, "model": candidate,
                "train_accuracy": train_metrics["accuracy"],
                "validation_accuracy": validation_metrics["accuracy"]}
    del candidate   # serverless caps the total size of models held in memory

display(spark.createDataFrame(
    tuning_rows, "C double, regParam double, train_accuracy double, validation_accuracy double, validation_roc_auc double"))

best_model = best["model"]
print(f"Selected C = {best['C']} (regParam = {1.0 / (best['C'] * N_TRAIN):.3g}): "
      f"train accuracy {best['train_accuracy']:.4f}, validation accuracy {best['validation_accuracy']:.4f}")

C,regParam,train_accuracy,validation_accuracy,validation_roc_auc
0.001,0.15625,0.7859,0.7837,0.8751
0.01,0.015625,0.8306,0.8363,0.8993
0.1,0.0015625,0.8473,0.8544,0.9152
1.0,1.5625E-4,0.8533,0.8525,0.9219
10.0,1.5625E-5,0.8556,0.8531,0.9227
100.0,1.5625E-6,0.8548,0.8538,0.9227
1000.0,1.5625E-7,0.8547,0.8538,0.9227


Selected C = 0.1 (regParam = 0.00156): train accuracy 0.8473, validation accuracy 0.8544


## 4. Test-set evaluation

The selected model is scored once on the test drivers. Precision, recall and F1 are for the claim class, as in Phase 2.

In [0]:
test_predictions = best_model.transform(test_df)
test_metrics = evaluate_classifier(test_predictions, "label")

METRICS = ["accuracy", "precision", "recall", "f1", "roc_auc"]
display(spark.createDataFrame(
    [(m, round(test_metrics[m], 4)) for m in METRICS], "metric string, test_value double"))

metric,test_value
accuracy,0.847
precision,0.7907
recall,0.7256
f1,0.7568
roc_auc,0.9158


In [0]:
# Confusion matrix on the test set.
display(spark.createDataFrame(
    [("Actual: no claim", test_metrics["tn"], test_metrics["fp"]),
     ("Actual: claim", test_metrics["fn"], test_metrics["tp"])],
    "actual string, predicted_no_claim long, predicted_claim long"))

actual,predicted_no_claim,predicted_claim
Actual: no claim,1218,126
Actual: claim,180,476


## 5. Comparison with Phase 2

Phase 2 trained logistic regression twice, once per feature-selection method, and both are shown. The baseline predicts "no claim" for everyone. The Phase 2 values are copied from the Phase 2 report (test-set performance table).

In [0]:
PHASE2_LR = {
    "Phase 2 scikit-learn LR (RF feature set)":
        {"accuracy": 0.8515, "precision": 0.7900, "recall": 0.7454, "f1": 0.7671, "roc_auc": 0.9213},
    "Phase 2 scikit-learn LR (Stepwise feature set)":
        {"accuracy": 0.8515, "precision": 0.7890, "recall": 0.7470, "f1": 0.7674, "roc_auc": 0.9229},
}
test_total = sum(test_metrics[k] for k in ("tp", "fp", "tn", "fn"))
baseline = {"accuracy": (test_metrics["tn"] + test_metrics["fp"]) / test_total,
            "precision": 0.0, "recall": 0.0, "f1": 0.0, "roc_auc": 0.5}

comparison = {"Always 'no claim' (this test set)": baseline, **PHASE2_LR,
              "Phase 3 Spark MLlib LR": test_metrics}

display(spark.createDataFrame(
    [(name, *[round(float(values[m]), 4) for m in METRICS]) for name, values in comparison.items()],
    "model string, accuracy double, precision double, recall double, f1 double, roc_auc double"))

model,accuracy,precision,recall,f1,roc_auc
Always 'no claim' (this test set),0.672,0.0,0.0,0.0,0.5
Phase 2 scikit-learn LR (RF feature set),0.8515,0.79,0.7454,0.7671,0.9213
Phase 2 scikit-learn LR (Stepwise feature set),0.8515,0.789,0.747,0.7674,0.9229
Phase 3 Spark MLlib LR,0.847,0.7907,0.7256,0.7568,0.9158


In [0]:
# Difference: Spark MLlib minus each Phase 2 model (positive = Spark is higher).
display(spark.createDataFrame(
    [(m, *[round(test_metrics[m] - phase2[m], 4) for phase2 in PHASE2_LR.values()]) for m in METRICS],
    "metric string, spark_minus_phase2_rf_set double, spark_minus_phase2_stepwise_set double"))

metric,spark_minus_phase2_rf_set,spark_minus_phase2_stepwise_set
accuracy,-0.0045,-0.0045
precision,7.0E-4,0.0017
recall,-0.0198,-0.0214
f1,-0.0103,-0.0106
roc_auc,-0.0055,-0.0071


### Why the numbers can differ

The split, the target and the metrics are the same as in Phase 2. The implementations differ in these ways:

| Aspect | Phase 2 (scikit-learn) | Phase 3 (Spark MLlib) |
|---|---|---|
| Test drivers | 2,000 | The same 2,000 (check the split table above) |
| Features | 16 (Random Forest importance) or 15 (stepwise) selected from 29 | Every feature in the Gold table, no selection step |
| `INSURANCE_RISK` | Each count scaled to 0–1, then added (SV/22 + DUI/6 + PA/15) | Plain sum of the counts (Phase 1 formula), scaled in the pipeline |
| `CAR_AGE` | 3 levels; missing mileage imputed first | 4 levels: the 957 drivers with missing mileage are `unknown` (196 of them in test) |
| Encoded features | 29 | 30 (the extra one is `CAR_AGE_unknown`) |
| Missing `CREDIT_SCORE` / `ANNUAL_MILEAGE` | Predicted by linear regression from the 3 most correlated columns | Training median |
| Rows used to fit preprocessing | 8,000 (train + validation) | 6,400 (train) |
| Tuning | 5-fold cross-validation, 13 values of `C` | One validation split, 7 values of `C` |
| Optimizer | `saga`, up to 5,000 iterations | L-BFGS, up to 200 iterations |

Logistic regression has a single best solution for a given dataset and penalty, so the optimizer should matter least. The feature set and the imputation method change the inputs themselves and are the more likely sources of any gap. One percentage point of accuracy on this test set is 20 drivers, so small gaps should not be over-interpreted.

## 6. Coefficients and odds ratios

Because every feature is scaled to 0–1, the odds ratio `exp(coefficient)` compares the top of a feature's range with the bottom. For a category it compares that category with the reference category of the same feature. The Phase 2 odds ratios are listed next to the Spark ones for the features Phase 2 reported; a blank means Phase 2 did not report that feature or the Gold table names it differently.

`INSURANCE_RISK` is the sum of `SPEEDING_VIOLATIONS`, `DUIS` and `PAST_ACCIDENTS`, which are also inputs, so its coefficient cannot be read on its own. As in Phase 2, read the four driving-record features as a group.

In [0]:
import math

indexer_model, lr_model = best_model.stages[0], best_model.stages[-1]

# Rebuild the feature names in the order the assembler used: one-hot columns, then numeric.
feature_names = []
for column, labels in zip(CATEGORICAL_FEATURES, indexer_model.labelsArray):
    feature_names += [f"{column}_{label}" for label in list(labels)[:-1]]   # last label = reference
    print(f"Reference category for {column}: {list(labels)[-1]}")
feature_names += NUMERIC_FEATURES

coefficients = [float(value) for value in lr_model.coefficients.toArray()]
assert len(coefficients) == len(feature_names), "Feature names do not line up with the coefficients"

# Odds ratios from the Phase 2 report: (RF feature set, Stepwise feature set).
PHASE2_ODDS_RATIOS = {
    "ANNUAL_MILEAGE": (8.93, 6.48), "DRIVING_EXPERIENCE_Beginner": (5.95, 6.85),
    "VEHICLE_YEAR_before 2015": (6.95, 6.54), "POSTAL_CODE_92101": (None, 3.77),
    "POSTAL_CODE_32765": (2.94, 3.12), "GENDER_male": (2.91, 2.81),
    "MARRIED": (0.65, 0.62), "VEHICLE_OWNERSHIP": (0.15, 0.16),
    "DRIVING_EXPERIENCE_Expert": (0.12, 0.12),
}

display(spark.createDataFrame(
    sorted(
        [(name, round(value, 4), round(math.exp(value), 3), *PHASE2_ODDS_RATIOS.get(name, (None, None)))
         for name, value in zip(feature_names, coefficients)],
        key=lambda row: -abs(row[1])),
    "feature string, coefficient double, odds_ratio double, "
    "phase2_odds_ratio_rf_set double, phase2_odds_ratio_stepwise_set double"))
print(f"Intercept: {lr_model.intercept:.4f}")

Reference category for AGE: Adult
Reference category for GENDER: female
Reference category for RACE: majority
Reference category for DRIVING_EXPERIENCE: Advanced
Reference category for EDUCATION: high school
Reference category for INCOME: middle class
Reference category for VEHICLE_YEAR: after 2015
Reference category for POSTAL_CODE: 10238
Reference category for VEHICLE_TYPE: sedan
Reference category for CAR_AGE: Newer


feature,coefficient,odds_ratio,phase2_odds_ratio_rf_set,phase2_odds_ratio_stepwise_set
POSTAL_CODE_21217,2.8499,17.286,null,null
DRIVING_EXPERIENCE_Expert,-1.6144,0.199,0.12,0.12
VEHICLE_OWNERSHIP,-1.5797,0.206,0.15,0.16
DRIVING_EXPERIENCE_Beginner,1.5768,4.84,5.95,6.85
VEHICLE_YEAR_before 2015,1.4409,4.224,6.95,6.54
POSTAL_CODE_92101,0.9147,2.496,null,3.77
GENDER_male,0.8862,2.426,2.91,2.81
POSTAL_CODE_32765,0.8659,2.377,2.94,3.12
ANNUAL_MILEAGE,0.6097,1.84,8.93,6.48
INSURANCE_RISK,-0.4815,0.618,null,null


Intercept: -2.3805


## 7. Save and reload the model

The whole fitted pipeline (preprocessing and classifier) is saved to the `models` volume. It is then loaded back and scored on the test set again to confirm the saved copy gives the same results.

In [0]:
LR_MODEL_PATH = f"{MODEL_DIR}/logistic_regression"

best_model.write().overwrite().save(LR_MODEL_PATH)
print("Saved to", LR_MODEL_PATH)

loaded_model = PipelineModel.load(LR_MODEL_PATH)
reloaded_metrics = evaluate_classifier(loaded_model.transform(test_df), "label")

for name in ("tp", "fp", "tn", "fn"):
    assert reloaded_metrics[name] == test_metrics[name], "The reloaded model predicts differently"
assert abs(reloaded_metrics["roc_auc"] - test_metrics["roc_auc"]) < 1e-9
print(f"Reloaded model reproduces the test results: accuracy {reloaded_metrics['accuracy']:.4f}, "
      f"ROC-AUC {reloaded_metrics['roc_auc']:.4f}")

Saved to /Volumes/workspace/car_insurance/models/logistic_regression
Reloaded model reproduces the test results: accuracy 0.8470, ROC-AUC 0.9158


**To use the saved model in another notebook**

```python
from pyspark.ml import PipelineModel
model = PipelineModel.load("/Volumes/workspace/car_insurance/models/logistic_regression")
predictions = model.transform(new_drivers)   # same columns as the Gold feature table
```

**Result.** A logistic regression pipeline was fitted with Spark MLlib on the Phase 2 training drivers, tuned on the validation drivers, evaluated once on the Phase 2 test drivers, and saved. Sections 4 and 5 hold the numbers to report.